# 6.18 — Vanishing & Exploding Gradients

Vanishing and exploding gradients happen because backpropagation through a deep network multiplies many local derivatives. If most layerwise gains are below 1, the learning signal shrinks toward zero before it reaches early layers; if most are above 1, it can blow up into unstable updates. In this lesson, you will build the arithmetic from scratch in NumPy, inspect the products directly, and see why initialization, activation choice, normalization, and step size are all scale-control tools.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build vanishing and exploding gradients one idea at a time. Run each cell in order and read the printed intermediate values — every product, update, and scale check is visible. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vector products, and reproducible numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for small random demonstrations.

### 1. A tiny layer: affine signal, gate, and local derivative

A neural layer first forms an affine signal $z=w^\top x+b$, then applies a nonlinearity. The lesson's scratch pass uses $x=[1.5,-0.5]$, weights $[1.6,-0.1]$, and bias $0.300$, so every number can be checked by hand. ReLU keeps positive signals unchanged and kills negative ones; its derivative is therefore either 1 or 0, which means it either passes the gradient backward or blocks it.

In [ ]:
x_w = np.array([1.5, -0.5])          # two input features.
w_w = np.array([1.6, -0.1])          # two weights feeding one neuron.
b_w = 0.300                          # scalar bias.
z_w = float(w_w @ x_w + b_w)         # affine signal before the gate.
h_w = max(0.0, z_w)                  # ReLU activation.
relu_prime_w = 1.0 if z_w > 0 else 0.0  # local derivative of ReLU at z.
print("affine z:", round(z_w, 3))
print("ReLU h:", round(h_w, 3), " ReLU derivative:", relu_prime_w)
assert round(z_w, 3) == 2.750 and round(h_w, 3) == 2.750

▶ What you'll see: the affine score is 2.750, the ReLU output is also 2.750, and the local derivative is 1 because the unit is active.

In [ ]:
parts_w = np.array([w_w[0] * x_w[0], w_w[1] * x_w[1], b_w])  # visible pieces of z.
plt.figure(figsize=(4.4, 3))
plt.bar(["w0*x0", "w1*x1", "b"], parts_w, color=["teal", "orange", "gray"])
plt.axhline(0, color="black", linewidth=0.7)
plt.title("1: pieces of the affine signal")
plt.ylabel("contribution to z")
plt.show()

▶ What you'll see: the first feature contributes 2.4, the second contributes 0.05, and the bias contributes 0.3, summing to 2.75.

*Why it's done this way:* Backpropagation is local. Each layer only needs its own derivative, but deep learning composes many such local choices. A ReLU derivative of 1 preserves gradient scale at this unit; a derivative of 0 would make the upstream gradient vanish immediately along this path.

### 2. Backpropagation is a product of local gains

For a chain $h_0\to h_1\to\cdots\to h_L\to L$, the chain rule multiplies local derivatives: $\frac{\partial L}{\partial h_0}=\left(\prod_{\ell=1}^L s_\ell\right)\frac{\partial L}{\partial h_L}$. That is the whole mechanism. A single gain near 1 looks harmless, but repeating it many times makes exponential shrinkage or growth.

In [ ]:
gains_w = np.array([0.9, 0.8, 1.1, 0.7, 0.95])  # five layerwise derivative magnitudes.
grad_out_w = 2.0                                 # gradient arriving from the loss side.
product_w = float(np.prod(gains_w))              # chain-rule multiplier.
grad_in_w = product_w * grad_out_w               # gradient reaching the earliest state.
print("local gains:", gains_w)
print("product:", round(product_w, 4), " input gradient:", round(grad_in_w, 4))
assert round(product_w, 4) == 0.5267

▶ What you'll see: the five local gains multiply to about 0.527, so a gradient of 2.0 becomes about 1.053.

In [ ]:
prefix_w = grad_out_w * np.cumprod(gains_w[::-1])[::-1]  # gradient size after crossing suffixes of the chain.
plt.figure(figsize=(4.6, 3))
plt.plot(np.arange(len(prefix_w)), prefix_w, marker="o", color="purple")
plt.title("2: gradient after repeated local products")
plt.xlabel("earlier position in the chain")
plt.ylabel("gradient magnitude")
plt.show()

▶ What you'll see: the gradient changes step by step as each local factor is multiplied into the signal.

*Why it's done this way:* The chain rule is unavoidable for composed functions. The danger is not one bad layer; it is repeated multiplication. Products below 1 erase information geometrically, while products above 1 amplify noise and curvature geometrically.

### 3. Vanishing versus exploding as depth grows

Now hold the local gain constant so the depth effect is isolated. A gain of 0.8 becomes $0.8^L$, while a gain of 1.2 becomes $1.2^L$. At depth 30 those are not slightly different; they live on very different scales.

In [ ]:
depths_w = np.arange(1, 31)              # depths from 1 to 30.
vanish_w = 0.8 ** depths_w               # repeated below-one gain.
explode_w = 1.2 ** depths_w              # repeated above-one gain.
print("0.8^30:", round(float(vanish_w[-1]), 4))
print("1.2^30:", round(float(explode_w[-1]), 3))
assert round(float(vanish_w[-1]), 4) == 0.0012
assert round(float(explode_w[-1]), 3) == 237.376

▶ What you'll see: the vanishing product is about 0.0012, while the exploding product is about 237.376.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(depths_w, vanish_w, label="0.8^depth", color="steelblue")
plt.plot(depths_w, explode_w, label="1.2^depth", color="crimson")
plt.yscale("log")
plt.title("3: products separate exponentially")
plt.xlabel("depth")
plt.ylabel("gradient multiplier, log scale")
plt.legend()
plt.show()

▶ What you'll see: on a log scale, the below-one product slopes downward and the above-one product slopes upward.

*Why it's done this way:* Exponentials are the natural result of repeated multiplication. This is why deep networks care so much about keeping the typical layerwise gain near 1: depth turns small scale errors into training failures.

### 4. Activation derivatives can quietly shrink gradients

Sigmoid is useful as a squashing function, but its derivative is $\sigma(z)(1-\sigma(z))$, which is at most 0.25 and becomes tiny when $z$ is far from 0. ReLU does not saturate on the positive side, but it has derivative 0 on the negative side. Different nonlinearities therefore create different gradient pathways.

In [ ]:
z_grid_w = np.linspace(-6, 6, 301)                      # preactivation values.
sig_w = 1 / (1 + np.exp(-z_grid_w))                     # sigmoid activation.
sig_prime_w = sig_w * (1 - sig_w)                       # sigmoid derivative.
relu_prime_grid_w = (z_grid_w > 0).astype(float)        # ReLU derivative except exactly at 0.
print("max sigmoid derivative:", round(float(sig_prime_w.max()), 3))
print("sigmoid derivative at z=6:", round(float(sig_prime_w[-1]), 4))
assert round(float(sig_prime_w.max()), 3) == 0.25

▶ What you'll see: sigmoid's best possible derivative is 0.25, and it is almost flat at large positive inputs.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(z_grid_w, sig_prime_w, label="sigmoid derivative", color="purple")
plt.plot(z_grid_w, relu_prime_grid_w, label="ReLU derivative", color="teal", alpha=0.8)
plt.title("4: activation derivatives set local gain")
plt.xlabel("preactivation z")
plt.ylabel("local derivative")
plt.legend()
plt.show()

▶ What you'll see: sigmoid has a small bell-shaped derivative, while ReLU passes gradient as 1 on positive inputs and 0 on negative inputs.

*Why it's done this way:* The derivative of the activation is one factor in the chain-rule product. Saturating activations create many below-one factors, and inactive ReLUs create exact zeros; both can starve early layers of gradient.

### 5. Initialization controls signal and gradient variance

Weights also multiply signals and gradients. If each layer's weights are too small, activations and gradients contract; if too large, they expand. A scratch simulation with random matrices shows why scale-aware initialization divides by input dimension: it tries to keep variance from drifting as depth increases.

In [ ]:
rng_w = np.random.default_rng(0)
width_w = 128
layers_w = 30
x0_w = rng_w.normal(size=(width_w, 1))
scales_w = [0.05, np.sqrt(2 / width_w), 0.30]  # too small, He-like for ReLU, too large.
labels_w = ["too small", "He-like", "too large"]
print("He-like scale:", round(scales_w[1], 4))
assert round(scales_w[1], 4) == 0.125

▶ What you'll see: the He-like scale for width 128 is 0.125, much smaller than an arbitrary large weight scale.

In [ ]:
norms_by_scale_w = []
for scale_w in scales_w:
    h_tmp_w = x0_w.copy()
    norms_w = []
    for layer_w in range(layers_w):
        W_tmp_w = rng_w.normal(0, scale_w, size=(width_w, width_w))
        h_tmp_w = np.maximum(0, W_tmp_w @ h_tmp_w)
        norms_w.append(float(np.linalg.norm(h_tmp_w)))
    norms_by_scale_w.append(norms_w)
print("final activation norms:", [round(curve[-1], 3) for curve in norms_by_scale_w])

▶ What you'll see: small weights drive norms toward zero, large weights inflate norms, and scale-aware weights stay more controlled.

In [ ]:
plt.figure(figsize=(5, 3))
for curve_w, label_w in zip(norms_by_scale_w, labels_w):
    plt.plot(curve_w, label=label_w)
plt.yscale("log")
plt.title("5: activation scale through depth")
plt.xlabel("layer")
plt.ylabel("activation norm, log scale")
plt.legend()
plt.show()

▶ What you'll see: the curves separate by orders of magnitude, showing that weight scale is a training-design choice, not a cosmetic detail.

*Why it's done this way:* Variance-preserving initialization is a practical attempt to make the typical product of weight scale and activation derivative close to neutral. It cannot guarantee perfect gradients, but it prevents obvious exponential drift at the starting point.

### 6. Normalization and updates keep arithmetic in a usable range

The content block normalizes the score 2.750 with mean 1.000 and variance 0.250, giving $(2.750-1.000)/\sqrt{0.250+10^{-5}}\approx3.500$. Normalization does not remove the need for gradients, but it gives each layer a more predictable scale. Then the optimizer makes a small parameter move, such as $2.000-0.080\cdot1.800=1.856$.

In [ ]:
score_w = 2.750
mean_w = 1.000
var_w = 0.250
eps_w = 0.00001
normed_w = (score_w - mean_w) / np.sqrt(var_w + eps_w)
theta_w = 2.000
eta_w = 0.080
grad_w = 1.800
theta_new_w = theta_w - eta_w * grad_w
print("normalized score:", round(float(normed_w), 3))
print("parameter update:", round(theta_new_w, 3))
assert round(float(normed_w), 3) == 3.5 and round(theta_new_w, 3) == 1.856

▶ What you'll see: the normalized score is 3.500 and the parameter moves from 2.000 to 1.856.

In [ ]:
batch_scores_w = np.array([0.4, 1.0, 2.75, 4.2])
centered_w = (batch_scores_w - mean_w) / np.sqrt(var_w + eps_w)
plt.figure(figsize=(4.6, 3))
plt.bar(["0.4", "1.0", "2.75", "4.2"], centered_w, color="darkorange")
plt.axhline(0, color="black", linewidth=0.7)
plt.title("6: normalized values are deviations from scale")
plt.ylabel("normalized value")
plt.show()

▶ What you'll see: values below the mean become negative, the mean becomes 0, and high scores become positive deviations.

*Why it's done this way:* Normalization attacks the scale problem before the chain-rule product gets too large or too small. The optimizer then relies on gradients that are numerically meaningful; if the gradient is already vanished or exploded, the update formula itself cannot rescue learning.

### 7. Scores, softmax, and memory bookkeeping

A model often turns raw scores into probabilities, so unstable scores can also create unstable comparisons. With scores 2.750 and 0.400, softmax assigns probability $e^{2.750}/(e^{2.750}+e^{0.400})\approx0.913$. Separately, a tiny activation block with 4 vectors of length 128 in 32-bit floats uses $4\cdot128\cdot4/1024=2$ KB; deep networks repeat this bookkeeping across many layers.

In [ ]:
logits_w = np.array([2.750, 0.400])
exp_w = np.exp(logits_w)
prob_w = exp_w[0] / np.sum(exp_w)
bytes_w = 4 * 128 * 4
kb_w = bytes_w / 1024
print("exp values:", np.round(exp_w, 3))
print("softmax probability for score 2.75:", round(float(prob_w), 3))
print("activation memory KB:", round(kb_w, 3))
assert round(float(prob_w), 3) == 0.913 and round(kb_w, 3) == 2.0

▶ What you'll see: the high score receives probability 0.913, and the small activation block uses 2 KB.

In [ ]:
logit_gap_w = np.linspace(0, 10, 101)
prob_curve_w = 1 / (1 + np.exp(-logit_gap_w))
plt.figure(figsize=(4.8, 3))
plt.plot(logit_gap_w, prob_curve_w, color="teal")
plt.scatter([2.35], [prob_w], color="red", zorder=3)
plt.title("7: softmax confidence grows with logit gap")
plt.xlabel("score gap")
plt.ylabel("probability of larger score")
plt.show()

▶ What you'll see: probability saturates as the score gap grows, so very large logits can make gradients less informative.

*Why it's done this way:* Deep learning arithmetic is not just symbolic. Scale affects probabilities, gradients, and stored activations, so stable training means managing math and hardware together.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Each tiny example isolates one
> gradient-scale mechanic from the walkthrough, prints the intermediate values, draws one
> picture, and ends with an `assert` that pins the calculation.

### ✍️ Toy 1 · Affine signal sets a local gate

A neuron first sums weighted inputs and bias; ReLU then either passes the value and derivative or shuts them off.

In [ ]:
import numpy as np                              # arrays and dot products for this toy.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t1_x = np.array([1.0, -1.0, 2.0, 0.0, 1.0, -2.0])       # six input features.
t1_w = np.array([0.5, 0.25, 0.75, -1.0, 0.5, -0.25])   # six weights.
t1_b = 0.25                                     # scalar bias.
t1_products = t1_w * t1_x                       # weighted pieces                   # -> [0.5, -0.25, 1.5, -0.0, 0.5, 0.5]
t1_z = float(np.sum(t1_products) + t1_b)        # affine signal                     # -> 3.0
t1_h = max(0.0, t1_z)                           # ReLU output                       # -> 3.0
t1_relu_prime = 1.0 if t1_z > 0 else 0.0        # local derivative                  # -> 1.0
print("weighted pieces:", t1_products.tolist()) # -> [0.5, -0.25, 1.5, -0.0, 0.5, 0.5]
print("bias:", round(t1_b, 3))                  # -> 0.25
print("affine z:", round(t1_z, 3))              # -> 3.0
print("ReLU output:", round(t1_h, 3))           # -> 3.0
print("ReLU derivative:", t1_relu_prime)        # -> 1.0
assert round(t1_z, 3) == 3.0
assert t1_relu_prime == 1.0

plt.figure(figsize=(4.8, 2.8))
plt.bar(["w0x0", "w1x1", "w2x2", "w3x3", "w4x4", "w5x5", "b"], np.append(t1_products, t1_b), color="teal")
plt.axhline(0, color="black", linewidth=0.7)
plt.ylabel("contribution")
plt.title("Toy 1 · affine pieces sum to z")
plt.show()

▶ What you'll see: the positive affine sum makes ReLU pass both activation and gradient.

### ✍️ Toy 2 · Chain rule multiplies local gains

Backpropagation through a chain multiplies every local gain into the gradient arriving from the loss.

In [ ]:
import numpy as np                              # arrays and products for this toy.

t2_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t2_gains = np.array([0.9, 0.8, 1.1, 0.7, 0.95, 0.85])  # six local derivative magnitudes.
t2_grad_out = 2.0                               # gradient at the end of the chain.
t2_product = float(np.prod(t2_gains))           # chain-rule multiplier             # -> 0.4477
t2_grad_in = t2_product * t2_grad_out           # gradient at the start             # -> 0.8954
t2_suffix = t2_grad_out * np.cumprod(t2_gains[::-1])[::-1]  # after crossing suffixes # -> [0.895, 0.995, 1.244, 1.13, 1.615, 1.7]
print("local gains:", t2_gains.tolist())       # -> [0.9, 0.8, 1.1, 0.7, 0.95, 0.85]
print("product:", round(t2_product, 4))         # -> 0.4477
print("input gradient:", round(t2_grad_in, 4))  # -> 0.8954
print("suffix gradients:", np.round(t2_suffix, 3).tolist())  # -> [0.895, 0.995, 1.244, 1.13, 1.615, 1.7]
assert round(t2_product, 4) == 0.4477
assert round(t2_grad_in, 4) == 0.8954

plt.figure(figsize=(4.8, 2.8))
plt.plot(np.arange(t2_suffix.size), t2_suffix, marker="o", color="purple")
plt.xlabel("earlier chain position")
plt.ylabel("gradient magnitude")
plt.title("Toy 2 · products change gradient scale")
plt.show()

▶ What you'll see: each extra local derivative changes the magnitude carried to earlier positions.

### ✍️ Toy 3 · Depth turns small gain errors exponential

A below-one gain shrinks with depth, while an above-one gain grows with depth.

In [ ]:
import numpy as np                              # arrays and powers for this toy.

t3_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t3_depths = np.arange(1, 13)                    # depths from 1 to 12.
t3_vanish = 0.8 ** t3_depths                    # repeated below-one gain           # -> final 0.0687
t3_explode = 1.2 ** t3_depths                   # repeated above-one gain           # -> final 8.9161
print("depths:", t3_depths.tolist())           # -> [1, 2, ..., 12]
print("0.8^depth:", np.round(t3_vanish, 4).tolist())   # -> [0.8, 0.64, ..., 0.0687]
print("1.2^depth:", np.round(t3_explode, 3).tolist())  # -> [1.2, 1.44, ..., 8.916]
print("final vanish/explode:", round(float(t3_vanish[-1]), 4), round(float(t3_explode[-1]), 3))  # -> 0.0687 8.916
assert round(float(t3_vanish[-1]), 4) == 0.0687
assert round(float(t3_explode[-1]), 3) == 8.916

plt.figure(figsize=(4.8, 2.8))
plt.plot(t3_depths, t3_vanish, marker="o", color="steelblue", label="0.8^L")
plt.plot(t3_depths, t3_explode, marker="s", color="crimson", label="1.2^L")
plt.yscale("log")
plt.xlabel("depth")
plt.ylabel("multiplier")
plt.title("Toy 3 · repeated products separate")
plt.legend()
plt.show()

▶ What you'll see: the log plot shows one curve falling and the other rising with depth.

### ✍️ Toy 4 · Activation derivatives control local gain

Sigmoid derivatives are at most 0.25 and shrink in saturation; ReLU derivatives are either 0 or 1 away from zero.

In [ ]:
import numpy as np                              # arrays and activation formulas for this toy.

t4_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t4_z = np.array([-4., -2., 0., 1., 2., 4.])     # six preactivation values.
t4_sigmoid = 1 / (1 + np.exp(-t4_z))            # sigmoid activation                # -> [0.018, 0.119, 0.5, 0.731, 0.881, 0.982]
t4_sigmoid_prime = t4_sigmoid * (1 - t4_sigmoid)  # sigmoid derivative              # -> [0.018, 0.105, 0.25, 0.197, 0.105, 0.018]
t4_relu_prime = (t4_z > 0).astype(float)        # ReLU derivative                  # -> [0.0, 0.0, 0.0, 1.0, 1.0, 1.0]
print("z values:", t4_z.tolist())              # -> [-4.0, -2.0, 0.0, 1.0, 2.0, 4.0]
print("sigmoid:", np.round(t4_sigmoid, 3).tolist())          # -> [0.018, 0.119, 0.5, 0.731, 0.881, 0.982]
print("sigmoid derivative:", np.round(t4_sigmoid_prime, 3).tolist())  # -> [0.018, 0.105, 0.25, 0.197, 0.105, 0.018]
print("ReLU derivative:", t4_relu_prime.tolist())             # -> [0.0, 0.0, 0.0, 1.0, 1.0, 1.0]
assert round(float(np.max(t4_sigmoid_prime)), 3) == 0.25
assert np.allclose(t4_relu_prime[-3:], 1.0)

plt.figure(figsize=(4.8, 2.8))
plt.plot(t4_z, t4_sigmoid_prime, marker="o", color="purple", label="sigmoid'")
plt.step(t4_z, t4_relu_prime, where="mid", color="teal", label="ReLU'")
plt.xlabel("z")
plt.ylabel("local derivative")
plt.title("Toy 4 · activation derivative is a gain")
plt.legend()
plt.show()

▶ What you'll see: sigmoid's derivative is small except near zero, while ReLU either blocks or passes gradient.

### ✍️ Toy 5 · Initialization scale predicts variance drift

With fan-in 6, the variance multiplier is `fan_in * scale²`; too small decays, scale-aware stays near one, and too large explodes.

In [ ]:
import numpy as np                              # arrays and variance multipliers for this toy.

t5_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t5_fan_in = 6                                   # six inputs into a layer.
t5_scales = np.array([0.2, 1 / np.sqrt(t5_fan_in), 0.8])  # small, scale-aware, large.
t5_multipliers = t5_fan_in * t5_scales ** 2     # variance multiplier per layer     # -> [0.24, 1.0, 3.84]
t5_depths = np.arange(1, 7)                     # six layers.
t5_curves = t5_multipliers[:, None] ** t5_depths[None, :]  # cumulative variance effect.
print("scales:", np.round(t5_scales, 3).tolist())          # -> [0.2, 0.408, 0.8]
print("variance multipliers:", np.round(t5_multipliers, 3).tolist())  # -> [0.24, 1.0, 3.84]
print("final variance factors:", np.round(t5_curves[:, -1], 3).tolist())  # -> [0.0, 1.0, 3206.176]
assert round(float(t5_multipliers[1]), 3) == 1.0
assert t5_curves[0, -1] < 0.001
assert t5_curves[2, -1] > 3000

plt.figure(figsize=(4.8, 2.8))
for t5_curve, t5_label in zip(t5_curves, ["small", "scale-aware", "large"]):
    plt.plot(t5_depths, t5_curve, marker="o", label=t5_label)
plt.yscale("log")
plt.xlabel("layer")
plt.ylabel("variance factor")
plt.title("Toy 5 · scale compounds through depth")
plt.legend()
plt.show()

▶ What you'll see: the small scale collapses variance, the large scale explodes it, and the scale-aware line stays flat.

### ✍️ Toy 6 · Normalization converts scores to deviations

Normalization makes raw scores comparable by subtracting a mean and dividing by a standard deviation.

In [ ]:
import numpy as np                              # arrays and normalization formula for this toy.

t6_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t6_scores = np.array([0.4, 1.0, 2.75, 4.2, -0.5, 1.6])  # six raw scores.
t6_mean = 1.0                                   # reference mean.
t6_var = 0.25                                   # reference variance.
t6_eps = 1e-5                                   # numerical guard.
t6_centered = t6_scores - t6_mean               # deviations from mean              # -> [-0.6, 0.0, 1.75, 3.2, -1.5, 0.6]
t6_normed = t6_centered / np.sqrt(t6_var + t6_eps)      # normalized deviations       # -> [-1.2, 0.0, 3.5, 6.4, -3.0, 1.2]
print("scores:", t6_scores.tolist())           # -> [0.4, 1.0, 2.75, 4.2, -0.5, 1.6]
print("centered:", np.round(t6_centered, 3).tolist())   # -> [-0.6, 0.0, 1.75, 3.2, -1.5, 0.6]
print("normalized:", np.round(t6_normed, 3).tolist())   # -> [-1.2, 0.0, 3.5, 6.4, -3.0, 1.2]
assert round(float(t6_normed[2]), 3) == 3.5

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t6_scores.size), t6_normed, color="darkorange")
plt.axhline(0, color="black", linewidth=0.7)
plt.xlabel("score index")
plt.ylabel("normalized value")
plt.title("Toy 6 · normalization reveals scale")
plt.show()

▶ What you'll see: scores above the mean become positive deviations and scores below it become negative.

### ✍️ Toy 7 · Learning rate turns gradients into updates

Even after scale control, the optimizer still uses `theta - eta * gradient` to make an actual parameter move.

In [ ]:
import numpy as np                              # arrays and vector updates for this toy.

t7_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t7_theta = np.array([2.0, -1.0, 0.5, 3.0, -2.0, 1.0])    # six parameters.
t7_grad = np.array([1.8, -0.5, 0.25, 2.0, -1.0, 0.75])   # six gradients.
t7_eta = 0.08                                   # learning rate.
t7_step = t7_eta * t7_grad                      # amount subtracted                 # -> [0.144, -0.04, 0.02, 0.16, -0.08, 0.06]
t7_new = t7_theta - t7_step                     # updated parameters                # -> [1.856, -0.96, 0.48, 2.84, -1.92, 0.94]
print("theta:", t7_theta.tolist())             # -> [2.0, -1.0, 0.5, 3.0, -2.0, 1.0]
print("gradient:", t7_grad.tolist())           # -> [1.8, -0.5, 0.25, 2.0, -1.0, 0.75]
print("step eta*g:", np.round(t7_step, 3).tolist())      # -> [0.144, -0.04, 0.02, 0.16, -0.08, 0.06]
print("new theta:", np.round(t7_new, 3).tolist())        # -> [1.856, -0.96, 0.48, 2.84, -1.92, 0.94]
assert round(float(t7_new[0]), 3) == 1.856

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t7_step.size), t7_step, color="seagreen")
plt.axhline(0, color="black", linewidth=0.7)
plt.xlabel("parameter")
plt.ylabel("subtracted amount")
plt.title("Toy 7 · gradients become movements")
plt.show()

▶ What you'll see: positive gradients subtract from parameters, while negative gradients add to them.

### ✍️ Toy 8 · Softmax confidence follows score gaps

Large logits become exponentials and then probabilities, so score scale affects confidence.

In [ ]:
import numpy as np                              # arrays and stable softmax for this toy.

t8_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t8_logits = np.array([2.75, 0.4, 1.0, -1.0, 0.0, 2.0])  # six logits.
t8_shifted = t8_logits - np.max(t8_logits)      # subtract max before exp           # -> [0.0, -2.35, -1.75, -3.75, -2.75, -0.75]
t8_exp = np.exp(t8_shifted)                     # unnormalized evidence             # -> [1.0, 0.095, 0.174, 0.024, 0.064, 0.472]
t8_probs = t8_exp / np.sum(t8_exp)              # probabilities                     # -> [0.547, 0.052, 0.095, 0.013, 0.035, 0.258]
print("logits:", t8_logits.tolist())           # -> [2.75, 0.4, 1.0, -1.0, 0.0, 2.0]
print("shifted logits:", np.round(t8_shifted, 3).tolist())  # -> [0.0, -2.35, -1.75, -3.75, -2.75, -0.75]
print("exp evidence:", np.round(t8_exp, 3).tolist())        # -> [1.0, 0.095, 0.174, 0.024, 0.064, 0.472]
print("probabilities:", np.round(t8_probs, 3).tolist())     # -> [0.547, 0.052, 0.095, 0.013, 0.035, 0.258]
assert int(np.argmax(t8_probs)) == 0
assert np.allclose(np.sum(t8_probs), 1.0)

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t8_logits.size), t8_probs, color="teal")
plt.xlabel("class")
plt.ylabel("probability")
plt.title("Toy 8 · high logit gets most probability")
plt.show()

▶ What you'll see: the largest logit receives the most mass, and the second-largest still gets visible probability.

### ✍️ Toy 9 · Activation memory is repeated bookkeeping

Stored activations cost bytes, and deep models repeat that cost across many layers.

In [ ]:
import numpy as np                              # arrays and memory arithmetic for this toy.

t9_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t9_lengths = np.array([4, 4, 8, 8, 16, 16])     # six activation vector lengths.
t9_batch = 4                                    # four examples.
t9_bytes_per_float = 4                          # float32 bytes.
t9_floats = t9_lengths * t9_batch               # floats per layer                   # -> [16, 16, 32, 32, 64, 64]
t9_bytes = t9_floats * t9_bytes_per_float       # bytes per layer                    # -> [64, 64, 128, 128, 256, 256]
t9_total_bytes = int(np.sum(t9_bytes))          # total bytes                        # -> 896
t9_total_kb = t9_total_bytes / 1024             # total KB                           # -> 0.875
print("lengths:", t9_lengths.tolist())         # -> [4, 4, 8, 8, 16, 16]
print("floats per layer:", t9_floats.tolist()) # -> [16, 16, 32, 32, 64, 64]
print("bytes per layer:", t9_bytes.tolist())   # -> [64, 64, 128, 128, 256, 256]
print("total bytes:", t9_total_bytes)          # -> 896
print("total KB:", round(float(t9_total_kb), 3))  # -> 0.875
assert round(float(t9_total_kb), 3) == 0.875

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t9_lengths.size), t9_bytes / 1024, color="slateblue")
plt.xlabel("stored activation")
plt.ylabel("KB")
plt.title("Toy 9 · memory adds across layers")
plt.show()

▶ What you'll see: later wider activations use more memory, and total memory is the sum of all bars.


## 🛠️ Setup

In [ ]:
import numpy as np # Load NumPy for arrays, dot products, random simulations, and numerical checks.
import matplotlib.pyplot as plt # Load Matplotlib for small diagnostic plots.
np.random.seed(0) # Make stochastic examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Multiply local gains

**Goal.** Compute one chain-rule product, because the earliest gradient is the final gradient times every local derivative in between.

In [ ]:
gains_b1 = np.array([0.9, 0.8, 1.1, 0.7, 0.95]) # Define five local derivative magnitudes.
grad_tail_b1 = 2.0 # Define the gradient arriving from the loss side.
product_b1 = float(np.prod(gains_b1)) # Multiply local gains into one chain-rule scale factor.
grad_head_b1 = grad_tail_b1 * product_b1 # Scale the tail gradient to get the head gradient.
print("product:", round(product_b1, 4), "head gradient:", round(grad_head_b1, 4)) # Inspect the result.
assert round(product_b1, 4) == 0.5267 # Verify the concrete chain-rule product.
plt.figure(figsize=(4, 3)) # Create a compact gain plot.
plt.bar(np.arange(len(gains_b1)), gains_b1, color="teal") # Show each local derivative magnitude.
plt.axhline(1, color="black", linestyle="--") # Mark neutral gain.
plt.title("Basic 1: local gains") # Title the plot.
plt.xlabel("layer") # Label layers.
plt.ylabel("gain") # Label gain size.
plt.show() # Display the figure.

▶ What you'll see: several gains sit below 1, and the product shrinks the gradient from 2.0 to about 1.053.

👀 Takeaway: deep gradients are products, so every local scale factor matters.

### Basic 2 — See a vanishing product

**Goal.** Raise a below-one factor to increasing depths, because repeated multiplication by 0.8 makes gradients disappear exponentially.

In [ ]:
depths_b2 = np.arange(1, 21) # Define depths from 1 through 20.
mult_b2 = 0.8 ** depths_b2 # Compute the gradient multiplier at each depth.
print("0.8^10:", round(float(mult_b2[9]), 4), "0.8^20:", round(float(mult_b2[-1]), 4)) # Inspect two depths.
assert round(float(mult_b2[9]), 4) == 0.1074 # Verify the depth-10 multiplier.
plt.figure(figsize=(4, 3)) # Create a line plot.
plt.plot(depths_b2, mult_b2, marker="o", color="steelblue") # Plot decay with depth.
plt.title("Basic 2: vanishing product") # Title the plot.
plt.xlabel("depth") # Label the depth axis.
plt.ylabel("0.8^depth") # Label the multiplier.
plt.show() # Display the plot.

▶ What you'll see: the multiplier drops quickly, reaching about 0.0115 by depth 20.

👀 Takeaway: a modest below-one gain becomes tiny when depth repeats it.

### Basic 3 — See an exploding product

**Goal.** Raise an above-one factor to increasing depths, because repeated multiplication by 1.2 makes gradients grow exponentially.

In [ ]:
depths_b3 = np.arange(1, 21) # Define depths from 1 through 20.
mult_b3 = 1.2 ** depths_b3 # Compute the exploding multiplier at each depth.
print("1.2^10:", round(float(mult_b3[9]), 3), "1.2^20:", round(float(mult_b3[-1]), 3)) # Inspect two depths.
assert round(float(mult_b3[9]), 3) == 6.192 # Verify the depth-10 multiplier.
plt.figure(figsize=(4, 3)) # Create a line plot.
plt.plot(depths_b3, mult_b3, marker="o", color="crimson") # Plot growth with depth.
plt.title("Basic 3: exploding product") # Title the plot.
plt.xlabel("depth") # Label the depth axis.
plt.ylabel("1.2^depth") # Label the multiplier.
plt.show() # Display the plot.

▶ What you'll see: the multiplier rises from 1.2 to more than 38 by depth 20.

👀 Takeaway: a slightly high gain can make early-layer gradients dangerously large.

### Basic 4 — ReLU gate derivative

**Goal.** Compute ReLU outputs and derivatives, because inactive ReLUs send zero gradient backward.

In [ ]:
z_b4 = np.array([-2.0, -0.1, 0.0, 1.5, 2.75]) # Define representative preactivation values.
h_b4 = np.maximum(0, z_b4) # Apply ReLU to each value.
deriv_b4 = (z_b4 > 0).astype(float) # Compute the ReLU derivative away from zero.
print("ReLU outputs:", h_b4) # Inspect forward activations.
print("ReLU derivatives:", deriv_b4) # Inspect local backward gates.
assert int(np.sum(deriv_b4 == 0)) == 3 # Verify three values block gradients in this convention.
plt.figure(figsize=(4, 3)) # Create a derivative plot.
plt.bar([str(v) for v in z_b4], deriv_b4, color="darkorange") # Draw derivative by preactivation.
plt.title("Basic 4: ReLU backward gate") # Title the plot.
plt.xlabel("z") # Label preactivation values.
plt.ylabel("d ReLU / dz") # Label derivative.
plt.show() # Display the plot.

▶ What you'll see: negative and zero preactivations have derivative 0, while positive preactivations have derivative 1.

👀 Takeaway: ReLU helps positive paths keep scale, but dead paths vanish exactly.

### Basic 5 — Sigmoid derivative shrinks

**Goal.** Compute sigmoid derivatives, because saturation creates tiny local gains.

In [ ]:
z_b5 = np.array([-6.0, -2.0, 0.0, 2.0, 6.0]) # Define preactivations from saturated negative to saturated positive.
sig_b5 = 1 / (1 + np.exp(-z_b5)) # Compute sigmoid values.
deriv_b5 = sig_b5 * (1 - sig_b5) # Compute sigmoid derivatives.
print("sigmoid:", np.round(sig_b5, 3)) # Inspect activations.
print("derivative:", np.round(deriv_b5, 4)) # Inspect local gains.
assert round(float(deriv_b5[2]), 3) == 0.25 # Verify the maximum derivative at zero.
plt.figure(figsize=(4, 3)) # Create a compact bar plot.
plt.bar([str(v) for v in z_b5], deriv_b5, color="purple") # Show derivative size by preactivation.
plt.title("Basic 5: sigmoid local gains") # Title the plot.
plt.xlabel("z") # Label preactivation values.
plt.ylabel("sigmoid'(z)") # Label derivative values.
plt.show() # Display the plot.

▶ What you'll see: the derivative peaks at 0.25 near z=0 and is tiny near ±6.

👀 Takeaway: saturating activations insert many below-one factors into the gradient product.

### Basic 6 — One gradient-descent step

**Goal.** Apply $\theta\leftarrow\theta-\eta g$, because gradient scale matters only through the parameter move it causes.

In [ ]:
theta_b6 = 2.000 # Define the current scalar parameter.
eta_b6 = 0.080 # Define the learning rate.
grad_b6 = 1.800 # Define the scalar gradient.
step_b6 = eta_b6 * grad_b6 # Compute the amount subtracted from the parameter.
theta_new_b6 = theta_b6 - step_b6 # Apply one gradient-descent update.
print("step:", round(step_b6, 3), "new theta:", round(theta_new_b6, 3)) # Inspect the movement.
assert round(theta_new_b6, 3) == 1.856 # Verify the lesson update.
plt.figure(figsize=(4, 3)) # Create a before-after plot.
plt.bar(["before", "after"], [theta_b6, theta_new_b6], color=["gray", "teal"]) # Compare parameter values.
plt.title("Basic 6: one optimizer nudge") # Title the plot.
plt.ylabel("parameter value") # Label the value axis.
plt.show() # Display the plot.

▶ What you'll see: the parameter moves from 2.000 down to 1.856, a small reliable nudge.

👀 Takeaway: vanished gradients cause tiny nudges; exploded gradients cause oversized nudges.

### Basic 7 — Normalize one score

**Goal.** Standardize a signal with mean and variance, because normalization measures deviations on a controlled scale.

In [ ]:
score_b7 = 2.750 # Define the raw score from the scratch pass.
mean_b7 = 1.000 # Define the normalization mean.
var_b7 = 0.250 # Define the normalization variance.
eps_b7 = 0.00001 # Add a small epsilon to avoid division by zero.
normalized_b7 = (score_b7 - mean_b7) / np.sqrt(var_b7 + eps_b7) # Compute the normalized value.
print("normalized value:", round(float(normalized_b7), 3)) # Inspect the standardized score.
assert round(float(normalized_b7), 3) == 3.5 # Verify the lesson value.
plt.figure(figsize=(4, 3)) # Create a compact comparison plot.
plt.bar(["raw", "mean", "normalized"], [score_b7, mean_b7, normalized_b7], color=["orange", "gray", "teal"]) # Compare raw and normalized quantities.
plt.title("Basic 7: normalization arithmetic") # Title the plot.
plt.show() # Display the plot.

▶ What you'll see: the score is 3.5 standard-deviation units above the chosen mean.

👀 Takeaway: normalization makes scale explicit before gradients propagate through depth.

### Basic 8 — Softmax from two scores

**Goal.** Convert two scores into a probability, because losses usually compare logits through exponentials.

In [ ]:
logits_b8 = np.array([2.750, 0.400]) # Define the lesson score and baseline.
exp_b8 = np.exp(logits_b8) # Exponentiate both logits.
prob_b8 = exp_b8[0] / np.sum(exp_b8) # Compute the two-class softmax probability for the first logit.
print("exp(logits):", np.round(exp_b8, 3)) # Inspect exponential scores.
print("probability:", round(float(prob_b8), 3)) # Inspect the normalized comparison.
assert round(float(prob_b8), 3) == 0.913 # Verify the lesson softmax probability.
plt.figure(figsize=(4, 3)) # Create a probability bar chart.
plt.bar(["score 2.75", "score 0.40"], [prob_b8, 1 - prob_b8], color=["teal", "gray"]) # Show both class probabilities.
plt.title("Basic 8: two-score softmax") # Title the plot.
plt.ylabel("probability") # Label probability axis.
plt.show() # Display the plot.

▶ What you'll see: the larger score gets about 91.3% probability.

👀 Takeaway: logits with large gaps can saturate probabilities and weaken useful gradients.

### Basic 9 — Activation memory

**Goal.** Compute activation memory, because backpropagation stores intermediate values and depth multiplies the cost.

In [ ]:
vectors_b9 = 4 # Define how many activation vectors are stored.
length_b9 = 128 # Define the length of each vector.
bytes_per_float_b9 = 4 # Use 32-bit floats.
kb_b9 = vectors_b9 * length_b9 * bytes_per_float_b9 / 1024 # Convert bytes to kilobytes.
print("activation memory KB:", round(kb_b9, 3)) # Inspect memory cost.
assert round(kb_b9, 3) == 2.0 # Verify the lesson memory number.
plt.figure(figsize=(4, 3)) # Create a simple memory plot.
plt.bar(["activations"], [kb_b9], color="slateblue") # Show the memory amount.
plt.title("Basic 9: stored activation memory") # Title the plot.
plt.ylabel("KB") # Label memory axis.
plt.show() # Display the plot.

▶ What you'll see: even a tiny block uses 2 KB; real networks repeat this many times.

👀 Takeaway: stable deep learning is also memory bookkeeping, not just calculus.

### Basic 10 — Compare safe and unsafe update sizes

**Goal.** Compare updates from small, normal, and huge gradients, because gradient scale directly controls parameter movement.

In [ ]:
theta_b10 = 2.0 # Define a starting parameter.
eta_b10 = 0.08 # Define one learning rate.
grads_b10 = np.array([0.001, 1.8, 100.0]) # Define vanished, usable, and exploded gradients.
updates_b10 = eta_b10 * grads_b10 # Compute parameter movements.
new_thetas_b10 = theta_b10 - updates_b10 # Apply the update formula to each gradient.
print("updates:", np.round(updates_b10, 4)) # Inspect movement magnitudes.
print("new thetas:", np.round(new_thetas_b10, 4)) # Inspect resulting parameters.
assert round(float(updates_b10[1]), 3) == 0.144 # Verify the normal update size.
plt.figure(figsize=(4.5, 3)) # Create a comparison plot.
plt.bar(["vanished", "usable", "exploded"], updates_b10, color=["steelblue", "teal", "crimson"]) # Show update magnitudes.
plt.yscale("log") # Use log scale so all three are visible.
plt.title("Basic 10: update scale") # Title the plot.
plt.ylabel("|eta * gradient|, log scale") # Label update size.
plt.show() # Display the plot.

▶ What you'll see: the vanished update is nearly invisible, while the exploded update is orders of magnitude larger.

👀 Takeaway: vanishing and exploding gradients are optimizer problems because they become bad update sizes.

## 🟡 Easy

### Easy 1 — Backpropagate through a scalar deep chain

**Goal.** Build a scalar network $h_L=(a^L)x$ and compute its exact input gradient, because this is the cleanest possible vanishing/exploding demo.

In [ ]:
x_e1 = 1.0 # Define the scalar input.
depth_e1 = 12 # Choose a moderately deep scalar chain.
gains_e1 = np.array([0.85, 1.00, 1.15]) # Compare shrinking, neutral, and growing chains.
outputs_e1 = x_e1 * (gains_e1 ** depth_e1) # Forward output for h=a^L x.
grads_e1 = gains_e1 ** depth_e1 # Backward derivative dh/dx is the same product.
print("outputs:", np.round(outputs_e1, 4)) # Inspect final activations.
print("input gradients:", np.round(grads_e1, 4)) # Inspect exact gradients.
assert round(float(grads_e1[0]), 4) == 0.1422 # Verify the shrinking chain value.
plt.figure(figsize=(4.5, 3)) # Create a compact comparison plot.
plt.bar(["a=0.85", "a=1.00", "a=1.15"], grads_e1, color=["steelblue", "gray", "crimson"]) # Compare gradient products.
plt.title("Easy 1: scalar chain gradients") # Title the plot.
plt.ylabel("dh_L/dx") # Label derivative axis.
plt.show() # Display the plot.

▶ What you'll see: the below-one chain produces a small gradient, the neutral chain stays at 1, and the above-one chain grows.

👀 Takeaway: deep-gradient behavior can be understood from the scalar chain-rule product before adding matrices.

### Easy 2 — Matrix products can change gradient norm

**Goal.** Multiply a gradient by repeated weight matrices, because dense layers backpropagate through transposed weight products.

In [ ]:
rng_e2 = np.random.default_rng(2) # Create reproducible matrices.
width_e2 = 20 # Define vector width.
depth_e2 = 15 # Define number of layers.
scales_e2 = [0.05, 0.22, 0.45] # Compare small, moderate, and large matrix scales.
final_norms_e2 = [] # Store final gradient norms.
for scale_e2 in scales_e2: # Loop through scales.
    g_e2 = np.ones(width_e2) / np.sqrt(width_e2) # Start with unit-norm output gradient.
    for layer_e2 in range(depth_e2): # Backpropagate through random matrices.
        W_e2 = rng_e2.normal(0, scale_e2, size=(width_e2, width_e2)) # Create one random layer matrix.
        g_e2 = W_e2.T @ g_e2 # Apply the transposed matrix to the gradient.
    final_norms_e2.append(float(np.linalg.norm(g_e2))) # Store the final input-gradient norm.
print("final gradient norms:", np.round(final_norms_e2, 6)) # Inspect scale effects.
assert final_norms_e2[0] < final_norms_e2[1] < final_norms_e2[2] # Verify monotonic growth in this seeded demo.
plt.figure(figsize=(4.5, 3)) # Create a scale comparison plot.
plt.bar(["0.05", "0.22", "0.45"], final_norms_e2, color=["steelblue", "teal", "crimson"]) # Show final norms.
plt.yscale("log") # Use log scale for readability.
plt.title("Easy 2: matrix backprop scale") # Title the plot.
plt.xlabel("weight std") # Label scale choices.
plt.ylabel("input-gradient norm") # Label gradient norm.
plt.show() # Display the plot.

▶ What you'll see: small matrices nearly erase the gradient, while large matrices amplify it.

👀 Takeaway: in real dense networks, the chain product is a product of matrices, not just scalars.

### Easy 3 — Initialization sweep for forward variance

**Goal.** Track activation variance through random ReLU layers, because unstable forward scale usually predicts unstable backward scale too.

In [ ]:
rng_e3 = np.random.default_rng(3) # Create reproducible randomness.
width_e3 = 100 # Define layer width.
depth_e3 = 20 # Define number of layers.
x_e3 = rng_e3.normal(size=(width_e3, 200)) # Simulate a batch of 200 examples.
scales_e3 = np.array([0.05, np.sqrt(2 / width_e3), 0.25]) # Compare too small, He-like, and too large.
labels_e3 = ["small", "He-like", "large"] # Name the scale choices.
variance_curves_e3 = [] # Store variance by layer.
for scale_e3 in scales_e3: # Loop over initialization scales.
    h_e3 = x_e3.copy() # Reset the input batch.
    curve_e3 = [] # Store this scale's layer variances.
    for layer_e3 in range(depth_e3): # Run a forward pass through random ReLU layers.
        W_e3 = rng_e3.normal(0, scale_e3, size=(width_e3, width_e3)) # Create weights.
        h_e3 = np.maximum(0, W_e3 @ h_e3) # Apply affine map and ReLU gate.
        curve_e3.append(float(np.var(h_e3))) # Store activation variance.
    variance_curves_e3.append(curve_e3) # Store the curve.
print("final variances:", [round(c[-1], 4) for c in variance_curves_e3]) # Inspect final scale.

▶ What you'll see: the final variances separate strongly across initialization scales.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a variance-through-depth figure.
for curve_e3, label_e3 in zip(variance_curves_e3, labels_e3): # Plot each scale curve.
    plt.plot(curve_e3, label=label_e3) # Draw the layer-by-layer variance.
plt.yscale("log") # Use log scale to show shrinkage and growth.
plt.title("Easy 3: forward variance by initialization") # Title the plot.
plt.xlabel("layer") # Label layer axis.
plt.ylabel("activation variance") # Label variance axis.
plt.legend() # Show scale labels.
plt.show() # Display the plot.

▶ What you'll see: scale-aware initialization slows variance drift compared with obviously small or large weights.

👀 Takeaway: good initialization tries to keep both forward activations and backward gradients in a usable range.

### Easy 4 — Normalization changes effective gradient scale

**Goal.** Normalize a batch and compare gradients through the scale factor, because dividing by standard deviation rescales backward signals.

In [ ]:
scores_e4 = np.array([0.4, 1.0, 2.75, 4.2]) # Define a tiny batch of raw scores.
mean_e4 = float(np.mean(scores_e4)) # Compute batch mean.
var_e4 = float(np.var(scores_e4)) # Compute batch variance.
std_e4 = np.sqrt(var_e4 + 1e-5) # Compute stable standard deviation.
normed_e4 = (scores_e4 - mean_e4) / std_e4 # Normalize scores.
upstream_e4 = np.ones_like(scores_e4) # Use a unit upstream gradient for inspection.
approx_grad_e4 = upstream_e4 / std_e4 # Show the main scaling effect of normalization.
print("mean:", round(mean_e4, 3), "std:", round(float(std_e4), 3)) # Inspect scale statistics.
print("normalized:", np.round(normed_e4, 3)) # Inspect normalized values.
print("approx gradient scale:", round(float(approx_grad_e4[0]), 3)) # Inspect backward rescale.
assert round(float(np.mean(normed_e4)), 6) == 0.0 # Verify normalized mean.
plt.figure(figsize=(4.5, 3)) # Create a normalized-value plot.
plt.bar(np.arange(len(scores_e4)), normed_e4, color="darkorange") # Show normalized scores.
plt.axhline(0, color="black", linewidth=0.7) # Mark zero mean.
plt.title("Easy 4: batch normalization effect") # Title the plot.
plt.ylabel("normalized score") # Label normalized scale.
plt.show() # Display the plot.

▶ What you'll see: normalized scores are centered at zero, and the backward scale is tied to the batch standard deviation.

👀 Takeaway: normalization manages gradient scale by making layer inputs live on a predictable numeric scale.

### Easy 5 — Detect an unsafe gradient norm

**Goal.** Measure gradient norm before an update, because exploding gradients are often diagnosed by unusually large norms.

In [ ]:
grad_e5 = np.array([3.0, -4.0, 12.0]) # Define a three-parameter gradient.
eta_e5 = 0.05 # Define a learning rate.
norm_e5 = float(np.linalg.norm(grad_e5)) # Compute gradient L2 norm.
update_e5 = -eta_e5 * grad_e5 # Compute the parameter update vector.
update_norm_e5 = float(np.linalg.norm(update_e5)) # Compute update norm.
print("gradient norm:", round(norm_e5, 3), "update norm:", round(update_norm_e5, 3)) # Inspect safety diagnostics.
assert round(norm_e5, 3) == 13.0 # Verify the 3-4-12 triangle norm.
plt.figure(figsize=(4, 3)) # Create a component plot.
plt.bar(["g0", "g1", "g2"], grad_e5, color="crimson") # Show gradient components.
plt.axhline(0, color="black", linewidth=0.7) # Mark zero.
plt.title("Easy 5: gradient components") # Title the plot.
plt.ylabel("gradient value") # Label gradient axis.
plt.show() # Display the plot.

▶ What you'll see: the gradient norm is 13, so even a modest learning rate gives a sizeable update norm of 0.65.

👀 Takeaway: norm checks turn "exploding" from a vague word into a measurable training signal.

## 🔴 Advanced

### Advanced 1 — Compare depth and gain on one heatmap

**Goal.** Map gradient multipliers across many gains and depths, because vanishing/exploding is a two-variable scale problem.

In [ ]:
gains_a1 = np.linspace(0.7, 1.3, 61) # Define local gains from shrinking to growing.
depths_a1 = np.arange(1, 41) # Define depths from 1 to 40.
log10_mult_a1 = np.array([[d_a1 * np.log10(g_a1) for g_a1 in gains_a1] for d_a1 in depths_a1]) # Compute log10(gain^depth).
print("log10 multiplier at gain .8 depth 30:", round(float(30 * np.log10(0.8)), 3)) # Inspect a vanishing point.
print("log10 multiplier at gain 1.2 depth 30:", round(float(30 * np.log10(1.2)), 3)) # Inspect an exploding point.
assert round(float(30 * np.log10(0.8)), 3) == -2.907 # Verify the vanishing log scale.
plt.figure(figsize=(5, 3.5)) # Create a heatmap figure.
plt.imshow(log10_mult_a1, aspect="auto", origin="lower", extent=[gains_a1[0], gains_a1[-1], depths_a1[0], depths_a1[-1]], cmap="coolwarm") # Draw log multipliers.
plt.colorbar(label="log10 gradient multiplier") # Add color scale.
plt.axvline(1.0, color="black", linestyle="--") # Mark neutral gain.
plt.title("Advanced 1: depth × gain scale map") # Title the heatmap.
plt.xlabel("typical local gain") # Label gain axis.
plt.ylabel("depth") # Label depth axis.
plt.show() # Display the heatmap.

▶ What you'll see: the region below gain 1 turns blue with depth, while the region above gain 1 turns red.

👀 Takeaway: depth magnifies tiny deviations from neutral gain into orders-of-magnitude differences.

### Advanced 2 — Backpropagate through a tanh network by hand

**Goal.** Store activations and derivatives for a small tanh network, because manual backprop exposes exactly where gradients shrink.

In [ ]:
rng_a2 = np.random.default_rng(12) # Create reproducible weights.
width_a2 = 6 # Define hidden width.
depth_a2 = 8 # Define network depth.
W_list_a2 = [0.7 * rng_a2.normal(size=(width_a2, width_a2)) / np.sqrt(width_a2) for _ in range(depth_a2)] # Initialize moderate tanh weights.
h_a2 = rng_a2.normal(size=width_a2) # Define one input vector.
activations_a2 = [h_a2] # Store activations for backprop.
derivs_a2 = [] # Store tanh derivatives.
for W_a2 in W_list_a2: # Forward pass through all layers.
    z_a2 = W_a2 @ activations_a2[-1] # Compute preactivation.
    h_next_a2 = np.tanh(z_a2) # Apply tanh.
    activations_a2.append(h_next_a2) # Store activation.
    derivs_a2.append(1 - h_next_a2 ** 2) # Store local tanh derivative.
g_a2 = np.ones(width_a2) / np.sqrt(width_a2) # Define unit-norm output gradient.
norms_a2 = [float(np.linalg.norm(g_a2))] # Track gradient norms backward.
for layer_a2 in range(depth_a2 - 1, -1, -1): # Backpropagate from last layer to first.
    g_a2 = W_list_a2[layer_a2].T @ (derivs_a2[layer_a2] * g_a2) # Apply tanh derivative then matrix transpose.
    norms_a2.append(float(np.linalg.norm(g_a2))) # Store norm after this layer.
print("backward norms:", np.round(norms_a2, 4)) # Inspect gradient flow.
assert norms_a2[-1] < norms_a2[0] # Verify shrinkage in this seeded demo.
plt.figure(figsize=(5, 3)) # Create a gradient-flow plot.
plt.plot(np.arange(len(norms_a2)), norms_a2, marker="o", color="purple") # Plot norm after each backward step.
plt.title("Advanced 2: manual tanh backprop norms") # Title the plot.
plt.xlabel("backward step") # Label steps.
plt.ylabel("gradient norm") # Label norm.
plt.show() # Display the plot.

▶ What you'll see: the gradient norm generally shrinks as it moves through tanh derivatives and weight matrices.

👀 Takeaway: manual backprop makes the gradient product visible as alternating activation derivatives and matrix transposes.

### Advanced 3 — Residual connection as a gradient shortcut

**Goal.** Compare a plain chain with a residual-style chain, because adding an identity path gives gradients a route that is not only the repeated small transform.

In [ ]:
depth_a3 = 20 # Define chain length.
alpha_a3 = 0.05 # Define a small residual transform strength.
plain_gain_a3 = alpha_a3 ** depth_a3 # Gradient through a plain repeated small transform.
residual_gain_a3 = (1 + alpha_a3) ** depth_a3 # Gradient through repeated h + alpha h blocks.
print("plain gain:", plain_gain_a3) # Inspect plain product.
print("residual gain:", round(residual_gain_a3, 3)) # Inspect residual product.
assert plain_gain_a3 < 1e-20 and round(residual_gain_a3, 3) == 2.653 # Verify the contrast.
curves_a3 = np.vstack([alpha_a3 ** np.arange(1, depth_a3 + 1), (1 + alpha_a3) ** np.arange(1, depth_a3 + 1)]) # Build both curves.
plt.figure(figsize=(5, 3)) # Create a comparison plot.
plt.plot(curves_a3[0], label="plain alpha^depth", color="crimson") # Plot plain product.
plt.plot(curves_a3[1], label="residual (1+alpha)^depth", color="teal") # Plot residual product.
plt.yscale("log") # Use log scale for huge contrast.
plt.title("Advanced 3: residual shortcut intuition") # Title the plot.
plt.xlabel("depth") # Label depth.
plt.ylabel("gradient multiplier") # Label multiplier.
plt.legend() # Show labels.
plt.show() # Display the plot.

▶ What you'll see: the plain product vanishes almost instantly, while the residual path keeps a strong route backward.

👀 Takeaway: residual connections fight vanishing gradients by adding an identity term to the derivative path.

### Advanced 4 — Clipping an exploded update for diagnosis

**Goal.** Cap a large gradient norm without changing direction, because clipping is a common safety response when products explode.

In [ ]:
g_a4 = np.array([30.0, -40.0, 120.0]) # Define an exploded gradient direction.
clip_a4 = 10.0 # Define a maximum allowed norm.
norm_a4 = float(np.linalg.norm(g_a4)) # Compute raw gradient norm.
scale_a4 = min(1.0, clip_a4 / norm_a4) # Compute clipping multiplier.
g_clip_a4 = g_a4 * scale_a4 # Apply norm clipping.
cos_a4 = float(np.dot(g_a4, g_clip_a4) / (np.linalg.norm(g_a4) * np.linalg.norm(g_clip_a4))) # Check direction preservation.
print("raw norm:", round(norm_a4, 3), "scale:", round(scale_a4, 4), "clipped norm:", round(float(np.linalg.norm(g_clip_a4)), 3)) # Inspect clipping.
print("cosine direction match:", round(cos_a4, 3)) # Inspect direction preservation.
assert round(float(np.linalg.norm(g_clip_a4)), 3) == 10.0 and round(cos_a4, 3) == 1.0 # Verify clipped norm and direction.
plt.figure(figsize=(4.5, 3)) # Create a norm comparison plot.
plt.bar(["raw", "clipped"], [norm_a4, np.linalg.norm(g_clip_a4)], color=["crimson", "teal"]) # Compare norms.
plt.axhline(clip_a4, color="black", linestyle="--", label="clip threshold") # Mark threshold.
plt.title("Advanced 4: norm clipping") # Title the plot.
plt.ylabel("gradient norm") # Label norm axis.
plt.legend() # Show threshold label.
plt.show() # Display the plot.

▶ What you'll see: the norm falls from 130 to 10 while the cosine with the original direction stays 1.

👀 Takeaway: clipping does not fix why gradients exploded, but it can prevent one update from destroying training.

### Advanced 5 — Train a tiny deep linear model with stable and unstable scales

**Goal.** Simulate gradient descent on a deep scalar linear model, because the same product controls both prediction and parameter gradients.

In [ ]:
x_a5 = 1.0 # Define one input.
y_a5 = 1.0 # Define one target.
depth_a5 = 8 # Define number of scalar layers.
eta_a5 = 0.01 # Define learning rate.
inits_a5 = [0.7, 1.0, 1.3] # Compare shrinking, neutral, and growing initial weights.
loss_curves_a5 = [] # Store loss curves.
for init_a5 in inits_a5: # Train one scalar deep model per initialization.
    w_a5 = np.full(depth_a5, init_a5, dtype=float) # Initialize all layer weights equally.
    losses_a5 = [] # Store this run's losses.
    for step_a5 in range(60): # Run gradient descent steps.
        pred_a5 = x_a5 * np.prod(w_a5) # Forward pass through scalar product.
        loss_a5 = 0.5 * (pred_a5 - y_a5) ** 2 # Squared error loss.
        losses_a5.append(float(loss_a5)) # Save loss.
        for i_a5 in range(depth_a5): # Compute gradient for each scalar weight.
            grad_i_a5 = (pred_a5 - y_a5) * x_a5 * np.prod(np.delete(w_a5, i_a5)) # dL/dw_i for product model.
            w_a5[i_a5] -= eta_a5 * grad_i_a5 # Apply a gradient step.
        w_a5 = np.clip(w_a5, -3, 3) # Keep the educational demo finite.
    loss_curves_a5.append(losses_a5) # Store the run.
print("initial losses:", [round(c[0], 4) for c in loss_curves_a5]) # Inspect initial fit.
print("final losses:", [round(c[-1], 4) for c in loss_curves_a5]) # Inspect final fit.
assert loss_curves_a5[1][-1] < 1e-8 # Verify the neutral initialization is already optimal.
plt.figure(figsize=(5, 3)) # Create a training-curve plot.
for curve_a5, init_a5 in zip(loss_curves_a5, inits_a5): # Plot each initialization.
    plt.plot(curve_a5, label=f"init={init_a5}") # Draw loss curve.
plt.yscale("log") # Use log scale for loss.
plt.title("Advanced 5: deep scalar training scale") # Title the plot.
plt.xlabel("step") # Label optimization step.
plt.ylabel("loss, log scale") # Label loss.
plt.legend() # Show initialization labels.
plt.show() # Display the plot.

▶ What you'll see: the neutral initialization is stable, while too-small and too-large products start with very different gradient regimes.

👀 Takeaway: even a toy deep product shows why scale, depth, and learning rate must be designed together.